<a href="https://colab.research.google.com/github/husnullkhatimah/Proyek-Data-Wrangling/blob/main/TUGAS_2_TPSDW_DATA_PREPARATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import dan Pembacaan Data**

Dataset berhasil dimuat menggunakan pandas dan ditampilkan 5 data pertama untuk memahami struktur data.

**Interpretasi:** Data memiliki beberapa kolom dengan nilai tidak valid seperti tanda '?', yang kemudian diubah menjadi NaN agar dapat diproses lebih lanjut.

In [1]:
import pandas as pd
import numpy as np

# Memuat dataset
df = pd.read_csv('/content/data_tugas 2.csv')

# Mengganti karakter '?' dengan NaN agar terdeteksi sebagai missing value
df = df.replace('?', np.nan)

# Menampilkan 5 data teratas
df.head()

,normalized-losses,make,bahan-bakar,jumlah-pintu,body-style,wheel-bases,length,width,height
0,NaN,alfa-romero,gas,two,convertible,88.6,168.8,64.1,48.8
1,NaN,alfa-romero,gas,two,convertible,88.6,168.8,64.1,48.8
2,NaN,alfa-romero,gas,two,hatchback,94.5,171.2,65.5,52.4
3,164,audi,gas,four,sedan,99.8,176.6,66.2,54.3
4,164,audi,gas,four,sedan,99.4,176.6,66.4,54.3


# **Identifikasi Missing Value**

Dilakukan pengecekan jumlah missing value pada setiap kolom.

**Interpretasi :** Ditemukan bahwa terdapat beberapa kolom dengan jumlah missing value yang cukup tinggi. Kolom dengan missing value terbanyak menunjukkan bahwa kualitas data pada variabel tersebut perlu perhatian khusus.

In [2]:
# Menghitung jumlah missing value per variabel
missing_values = df.isnull().sum()

# Membuat tabel ringkasan
print("Tabel Jumlah Missing Value:")
print(missing_values)

# Mencari variabel dengan missing value terbanyak
max_missing_col = missing_values.idxmax()
max_missing_val = missing_values.max()

print(f"\nInterpretasi: Variabel yang memiliki data hilang paling banyak adalah '{max_missing_col}' dengan jumlah {max_missing_val} data.")

Tabel Jumlah Missing Value:
normalized-losses    41
make                  4
bahan-bakar          13
jumlah-pintu         18
body-style           12
wheel-bases           3
length                8
width                 5
height                4
dtype: int64

Interpretasi: Variabel yang memiliki data hilang paling banyak adalah 'normalized-losses' dengan jumlah 41 data.


# **Penanganan Data Kategorikal**

Missing value pada kolom kategorikal diisi dengan nilai "unknown".

**Interpretasi :** Pendekatan ini dilakukan karena:
*   Tidak memungkinkan menggunakan rata-rata
*   Menghindari kehilangan data
*   Mempertahankan struktur dataset

Distribusi kategori menunjukkan variasi data yang cukup beragam, meskipun terdapat kategori baru yaitu "unknown".

In [3]:
# Daftar variabel kategori
cat_cols = ['make', 'bahan-bakar', 'jumlah-pintu', 'body-style']

# Mengganti NaN dengan "unknown"
for col in cat_cols:
    df[col] = df[col].fillna('unknown')

# Menampilkan ringkasan jumlah kategori
print("Ringkasan Jumlah Kategori:")
for col in cat_cols:
    print(f"\nVariabel: {col}")
    print(df[col].value_counts())


Ringkasan Jumlah Kategori:

Variabel: make
make
toyota           32
mazda            17
nissan           17
honda            13
mitsubishi       12
volkswagen       12
volvo            11
subaru           11
peugot           11
dodge             9
mercedes-benz     8
bmw               8
plymouth          7
audi              7
saab              6
porsche           4
unknown           4
isuzu             4
alfa-romero       3
chevrolet         3
jaguar            3
renault           2
mercury           1
Name: count, dtype: int64

Variabel: bahan-bakar
bahan-bakar
gas        173
diesel      19
unknown     13
Name: count, dtype: int64

Variabel: jumlah-pintu
jumlah-pintu
four       101
two         86
unknown     18
Name: count, dtype: int64

Variabel: body-style
body-style
sedan          88
hatchback      67
wagon          24
unknown        12
hardtop         8
convertible     6
Name: count, dtype: int64


# **Penanganan Data Numerik**

Data numerik dikonversi ke format angka, kemudian missing value diisi dengan nilai rata-rata (mean).

**Interpretasi :** Penggunaan mean dipilih karena:


*   Menjaga distribusi data tetap stabil
*   Tidak mengubah pola data secara signifikan
Namun, metode ini sensitif terhadap outlier.

In [4]:
# Daftar variabel numerik
num_cols = ['length', 'width', 'height']

# Mengubah tipe data ke float (untuk memastikan perhitungan mean akurat)
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Hitung rata-rata dan isi missing value
means = df[num_cols].mean()
df[num_cols] = df[num_cols].fillna(means)

print("Nilai rata-rata yang digunakan untuk imputasi:")
print(means)

Nilai rata-rata yang digunakan untuk imputasi:
length    173.902538
width      65.931000
height     53.737313
dtype: float64


# **Normalisasi Data (Min-Max)**

Dilakukan normalisasi pada variabel tertentu ke skala 0–1.

**Interpretasi :** Hasil normalisasi menunjukkan bahwa:
*   Nilai minimum menjadi 0
*   Data menjadi lebih seragam

Hal ini penting untuk algoritma machine learning yang sensitif terhadap skala data.

In [5]:
# Rumus Min-Max: (x - min) / (max - min)
df['wheel-bases-normalized'] = (df['wheel-bases'] - df['wheel-bases'].min()) / (df['wheel-bases'].max() - df['wheel-bases'].min())

# Menampilkan perbandingan kolom asli dan hasil normalisasi
print(df[['wheel-bases', 'wheel-bases-normalized']].head())

   wheel-bases  wheel-bases-normalized
0         88.6                0.058309
1         88.6                0.058309
2         94.5                0.230321
3         99.8                0.384840
4         99.4                0.373178


# **Standarisasi Data (Z-Score)**

Data diubah menggunakan metode Z-score.

**Interpretasi :** Setelah standarisasi:
*   Rata-rata data menjadi mendekati 0
*   Standar deviasi menjadi 1

Hal ini memudahkan analisis statistik dan meningkatkan performa model tertentu seperti regresi dan clustering.

In [6]:
# Rumus Z-score: (x - mean) / std
for col in ['length', 'width', 'height']:
    df[f'{col}-zscore'] = (df[col] - df[col].mean()) / df[col].std()

# Menampilkan hasil standarisasi
print(df[['length', 'length-zscore', 'width', 'width-zscore', 'height', 'height-zscore']].head())

   length  length-zscore  width  width-zscore  height  height-zscore
0   168.8      -0.428526   64.1     -0.867630    48.8      -2.038094
1   168.8      -0.428526   64.1     -0.867630    48.8      -2.038094
2   171.2      -0.226967   65.5     -0.204232    52.4      -0.552035
3   176.6       0.226541   66.2      0.127467    54.3       0.232274
4   176.6       0.226541   66.4      0.222238    54.3       0.232274


# **Penyimpanan Data**

Dataset yang telah dibersihkan disimpan dalam file baru.

**Interpretasi :** Dataset akhir sudah:
*   Bebas dari missing value
*   Memiliki format konsisten
*   Siap digunakan untuk analisis lanjutan


In [7]:
df.to_csv('data_tugas_2_clean.csv', index=False)